# 00 · Verificación del sistema

Este notebook corre dentro de un contenedor con acceso a la GPU. Repite desde Python lo que ya revisamos a mano en la terminal: la GPU, el driver, CUDA y PyTorch. Al final compara la misma operación en CPU y en GPU.

Para ejecutar todo: menú **Run > Restart Kernel and Run All Cells**.

## 1. La GPU vista desde el contenedor

`nvidia-smi` es el mismo comando que usamos en la terminal. Que funcione aquí adentro quiere decir que Docker le está pasando la GPU al contenedor.

In [ ]:
!nvidia-smi

## 2. Driver y memoria

NVML es la biblioteca de la que `nvidia-smi` saca sus datos; desde Python se usa con `pynvml`. El panel de GPU de la interfaz web lee lo mismo.

In [ ]:
import pynvml


def text(value):
    # Según la versión de la biblioteca, los nombres llegan como bytes o como texto.
    return value.decode() if isinstance(value, bytes) else value


pynvml.nvmlInit()
print("Driver NVIDIA:", text(pynvml.nvmlSystemGetDriverVersion()))

for index in range(pynvml.nvmlDeviceGetCount()):
    handle = pynvml.nvmlDeviceGetHandleByIndex(index)
    print(f"GPU {index}:", text(pynvml.nvmlDeviceGetName(handle)))
    try:
        memory = pynvml.nvmlDeviceGetMemoryInfo(handle)
        print(f"  memoria de GPU: {memory.used / 2**30:.1f} de {memory.total / 2**30:.1f} GB en uso")
    except pynvml.NVMLError:
        print("  esta GPU no reporta memoria propia: usa la memoria del sistema")

En un equipo de memoria unificada, como la DGX Spark, la CPU y la GPU comparten la misma memoria. Ahí la cifra que importa es la del sistema:

In [ ]:
def system_memory_gb():
    fields = {}
    with open("/proc/meminfo") as f:
        for line in f:
            key, value = line.split(":", 1)
            fields[key] = int(value.split()[0])  # en kB
    total = fields["MemTotal"]
    return (total - fields["MemAvailable"]) / 2**20, total / 2**20


used, total = system_memory_gb()
print(f"Memoria del sistema: {used:.1f} de {total:.1f} GB en uso")

## 3. PyTorch y CUDA

El contenedor viene de NGC, el catálogo de NVIDIA (`nvcr.io`), y trae PyTorch compilado para esta máquina. Si `CUDA disponible` dice `True`, PyTorch puede usar la GPU.

In [ ]:
import platform

import torch

HAS_GPU = torch.cuda.is_available()

print("Arquitectura:    ", platform.machine())
print("PyTorch:         ", torch.__version__)
print("CUDA disponible: ", HAS_GPU)
if HAS_GPU:
    print("Versión de CUDA: ", torch.version.cuda)
    print("cuDNN:           ", torch.backends.cudnn.version())
    print("Dispositivo:     ", torch.cuda.get_device_name(0))
else:
    print("PyTorch no ve la GPU. Revisa que el contenedor se haya levantado con docker compose.")

## 4. CPU contra GPU

Multiplicamos dos matrices cuadradas, la operación básica de una red neuronal, primero en la CPU y luego en la GPU. Se repite varias veces y se toma el mejor tiempo, para no contar el arranque.

In [ ]:
import time

import pandas as pd


def time_matmul(device, n, repeats=4):
    a = torch.rand(n, n, device=device)
    b = torch.rand(n, n, device=device)
    best = float("inf")
    for _ in range(repeats):
        start = time.perf_counter()
        c = a @ b
        if device == "cuda":
            torch.cuda.synchronize()  # la GPU trabaja en segundo plano: hay que esperarla
        best = min(best, time.perf_counter() - start)
    del a, b, c
    return best * 1000  # milisegundos


SIZES = [1024, 2048, 4096]

rows = []
for n in SIZES:
    cpu_ms = time_matmul("cpu", n)
    gpu_ms = time_matmul("cuda", n) if HAS_GPU else None
    rows.append(
        {
            "matriz": f"{n} x {n}",
            "CPU (ms)": cpu_ms,
            "GPU (ms)": gpu_ms,
            "veces más rápida": cpu_ms / gpu_ms if gpu_ms else None,
        }
    )

results = pd.DataFrame(rows)
results.round(2)

In [ ]:
import matplotlib.pyplot as plt

if HAS_GPU:
    fig, ax = plt.subplots(figsize=(7, 3.6))
    bars = ax.bar(results["matriz"], results["veces más rápida"], width=0.3, color="#059669")
    ax.bar_label(bars, labels=[f"{value:.0f}x" for value in results["veces más rápida"]], padding=4, fontsize=12)
    ax.set_title("Cuántas veces más rápida es la GPU que la CPU", loc="left", fontsize=13, fontweight="bold")
    ax.set_xlabel("Tamaño de la matriz")
    ax.set_yticks([])
    ax.margins(y=0.2)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    plt.show()
else:
    print("Sin GPU no hay comparación que graficar.")

La ventaja cambia con el tamaño de la matriz y con la máquina. Prueba otros tamaños en `SIZES` y compara con lo que obtuvo el resto del grupo.

## 5. Devolver la memoria

Lo que este notebook reservó en la GPU se libera aquí. Al terminar, cierra también el kernel (**Kernel > Shut Down Kernel**) para que esa memoria quede disponible para los modelos.

In [ ]:
if HAS_GPU:
    torch.cuda.empty_cache()
    print(f"Memoria de GPU que aún reserva este notebook: {torch.cuda.memory_reserved() / 2**20:.0f} MB")

used, total = system_memory_gb()
print(f"Memoria del sistema: {used:.1f} de {total:.1f} GB en uso")